# Notebook 2 — Python Tutorial: cellAdmix on CosMx Data

This notebook demonstrates every public function in py-cellAdmix using
the bundled CosMx NSCLC dataset.

**Audience**: End users new to the algorithm who want a Python walkthrough.

## What this package does

cellAdmix evaluates and corrects cell admixtures in imaging-based spatial
transcriptomics data. It uses NMF factorization of KNN neighborhood count
matrices to identify contamination factors, then applies CRF-based molecule
segmentation and Bayesian scoring to estimate per-cell admixture fractions.

**Input**: Molecule-level spatial transcriptomics data with x, y, z coordinates,
gene labels, cell assignments, and cell type annotations.

**Output**: Admixture scores per cell, contamination probabilities per gene,
and factor assignments per molecule.

## Install + Import

In [ ]:
# pip install pycelladmix

import celladmix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

print(f'py-cellAdmix version: {celladmix.__version__}')

## Load Demo Data

In [ ]:
# Load the bundled CosMx NSCLC fixture
df = pd.read_csv('../data/fixture_small_df.csv')
cell_meta = pd.read_csv('../data/fixture_small_cell_meta.csv')

print(f'Molecules: {len(df)}')
print(f'Cells: {df["cell"].nunique()}')
print(f'Genes: {df["gene"].nunique()}')
print(f'Cell types: {cell_meta["celltype"].nunique()}')
print()
print('Cell type distribution:')
print(cell_meta['celltype'].value_counts().to_string())

In [ ]:
# Plot spatial distribution
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['x'], df['y'], c=df['cell'].astype('category').cat.codes,
                     s=1, alpha=0.5, cmap='tab20')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Molecule Spatial Distribution')
plt.show()

## 1. KNN Adjacency Matrix

Builds a k-nearest-neighbor graph from spatial coordinates.

In [ ]:
# R equivalent: knn.adjacency.matrix(df, k=10)
adj = celladmix.knn_adjacency_matrix(df, k=10)

print(f'Adjacency matrix shape: {adj.shape}')
print(f'Non-zero entries: {adj.nnz}')
print(f'Row sums (should all be 10): {np.unique(np.asarray(adj.sum(axis=1)).ravel())}')

## 2. KNN Count Matrix

Counts genes per molecule's KNN neighborhood.

In [ ]:
# R equivalent: knn.count.matrix(df, k=10)
ncm, genes = celladmix.knn_count_matrix(df, k=10)

print(f'KNN count matrix shape: {ncm.shape}')
print(f'Genes: {len(genes)}')
print(f'Mean counts per molecule: {np.asarray(ncm.sum(axis=1)).ravel().mean():.1f}')

## 3. KNN Counts for All Cells

Computes KNN count matrices for molecules in all cells.

In [ ]:
# R equivalent: get.knn.counts.all(df, h=20)
knn_counts = celladmix.get_knn_counts_all(df, h=20)

print(f'KNN counts shape: {knn_counts.shape}')
print(f'Columns: {knn_counts.columns.tolist()[:10]}...')

## 4. Cell Adjacency

Estimates cell adjacency using Delaunay triangulation.

In [ ]:
# R equivalent: estimate.cell.adjacency(df)
cell_adj = celladmix.estimate_cell_adjacency(df, random_shift=1e-2)

print(f'Cell adjacency edges: {len(cell_adj)}')
print(f'Columns: {cell_adj.columns.tolist()}')
print(cell_adj.head())

## 5. Cell Type Adjacency

Computes mean adjacency counts between cell types.

In [ ]:
# R equivalent: estimate.cell.type.adjacency(cell.adj)
ct_adj = celladmix.estimate_cell_type_adjacency(cell_adj)

print(f'Cell type adjacency matrix shape: {ct_adj.shape}')
print(ct_adj)

## 6. Gene Probabilities Per Type

Estimates gene expression probabilities per cell type.

In [ ]:
# R equivalent: estimate.gene.prob.per.type(cm, annot)
gene_factors = pd.Categorical(df['gene'])
cell_factors = pd.Categorical(df['cell'])
cm = sparse.csr_matrix(
    (np.ones(len(df)), (gene_factors.codes, cell_factors.codes)),
    shape=(len(gene_factors.categories), len(cell_factors.categories))
)
cm_df = pd.DataFrame(cm.toarray(), index=gene_factors.categories, columns=cell_factors.categories)

annot = cell_meta.set_index('cell')['celltype']
gene_probs = celladmix.estimate_gene_prob_per_type(cm_df, annot)

print(f'Gene probabilities shape: {gene_probs.shape}')
print(f'Cell types: {gene_probs.index.tolist()}')

In [ ]:
# Heatmap of top variable genes
gene_var = gene_probs.var(axis=0).sort_values(ascending=False)
top_genes = gene_var.head(20).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(gene_probs[top_genes], cmap='YlOrRd', ax=ax)
ax.set_title('Top 20 Variable Genes: Expression Probability by Cell Type')
ax.set_xlabel('Genes')
ax.set_ylabel('Cell Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. NMF Factorization

Runs KNN-based NMF to identify contamination factors.

In [ ]:
# R equivalent: run.knn.nmf(df, k=3, h=20)
nmf_res = celladmix.run_knn_nmf(df, k=3, h=20)

print(f'NMF H shape (factors x genes): {nmf_res["H"].shape}')
print(f'NMF W shape (molecules x factors): {nmf_res["W"].shape}')
print(f'All H non-negative: {np.all(nmf_res["H"] >= 0)}')
print(f'All W non-negative: {np.all(nmf_res["W"] >= 0)}')

In [ ]:
# Plot NMF factor loadings
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, ax in enumerate(axes):
    h = nmf_res['H'][i]
    fracs = h / h.sum() if h.sum() > 0 else h
    top_idx = np.argsort(fracs)[-10:][::-1]
    top_genes = [nmf_res['feature_names'][j] for j in top_idx]
    top_fracs = fracs[top_idx]
    
    ax.barh(range(len(top_genes)), top_fracs, color='steelblue')
    ax.set_yticks(range(len(top_genes)))
    ax.set_yticklabels(top_genes, fontsize=7)
    ax.set_xlabel('Loading fraction')
    ax.set_title(f'Factor {i+1}')
    ax.invert_yaxis()

plt.suptitle('Top 10 Genes per NMF Factor')
plt.tight_layout()
plt.show()

## 8. Sparse Correlation

Computes correlation matrix efficiently for sparse data.

In [ ]:
# R equivalent: sparse.cor(x)
cor_mat = celladmix.sparse_cor(cm_df.values)

print(f'Correlation matrix shape: {cor_mat.shape}')
print(f'Diagonal (should be 1): {np.diag(cor_mat)[:5]}')
print(f'Value range: [{cor_mat.min():.3f}, {cor_mat.max():.3f}]')

## 9. Normalize Images

Normalizes fluorescence images for membrane tests.

In [ ]:
# R equivalent: normalize.images(im)
test_im = np.random.rand(3, 100, 100) * 255
norm_im = celladmix.normalize_images(test_im)

print(f'Input range: [{test_im.min():.1f}, {test_im.max():.1f}]')
print(f'Output range: [{norm_im.min():.6f}, {norm_im.max():.6f}]')

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(norm_im[i], cmap='gray')
    ax.set_title(f'Channel {i}')
    ax.axis('off')
plt.suptitle('Normalized Image')
plt.tight_layout()
plt.show()

## 10. Class API Mirror

The same workflow using the `CellAdmix` class with method chaining.

In [ ]:
# Class API
ca = celladmix.CellAdmix(molecule_df=df)

# Run NMF
ca.run_knn_nmf(k=3, h=20)
print('NMF results:', ca.nmf_results['H'].shape)

# Results are stored as properties
print('Feature names:', ca.nmf_results['feature_names'][:5])

## Common Pitfalls / FAQ

### 1. Row vs Column Convention
R uses genes × cells convention for count matrices, while Python/AnnData
uses cells × genes. The `estimate_gene_prob_per_type` function expects
genes × cells (matching R).

### 2. Base-1 vs Base-0 Indexing
R uses 1-based indexing. CRF factor labels are stored as 1-indexed to
match R output. When comparing with R results, no offset is needed.

### 3. Column Names
The R package uses both `cell_type` and `celltype` in different functions.
The Python port accepts both names for compatibility.

## Where to Go Next

- [README.md](../README.md) — Full documentation
- [MATH.md](../MATH.md) — Algorithm derivations
- [compare_R_vs_Python.ipynb](compare_R_vs_Python.ipynb) — Pipeline parity validation
- [cellAdmix R package](https://github.com/KharchenkoLab/cellAdmix) — Upstream R source